In [12]:
#!/usr/bin/env python3
"""Contour map generation - imports and environment detection."""

import os
import sys
import json
import gzip
from pathlib import Path
from datetime import datetime
from typing import Optional, Tuple

# NumPy for array operations
try:
    import numpy as np
    print("NumPy loaded")
except ImportError:
    print("Installing NumPy...")
    %pip install -q numpy
    import numpy as np
    print("NumPy installed and loaded")

# Matplotlib for contour generation
try:
    import matplotlib
    matplotlib.use('Agg')  # Non-interactive backend for file output
    import matplotlib.pyplot as plt
    from matplotlib.colors import LinearSegmentedColormap
    print("Matplotlib loaded")
except ImportError:
    print("Installing Matplotlib...")
    %pip install -q matplotlib
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from matplotlib.colors import LinearSegmentedColormap
    print("Matplotlib installed and loaded")

# SciPy for interpolation
try:
    from scipy.ndimage import zoom
    print("SciPy loaded")
except ImportError:
    print("Installing SciPy...")
    %pip install -q scipy
    from scipy.ndimage import zoom
    print("SciPy installed and loaded")

# Pillow for reading minimap dimensions
try:
    from PIL import Image
    print("Pillow loaded")
except ImportError:
    print("Installing Pillow...")
    %pip install -q Pillow
    from PIL import Image
    print("Pillow installed and loaded")

# Detect environment
try:
    IN_COLAB = 'google.colab' in str(get_ipython())
except:
    IN_COLAB = False

print(f"\n{'='*70}")
print(f"Environment: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print(f"{'='*70}\n")

# Verify repository structure
repo_root = Path.cwd()
if not (repo_root / 'processed_maps').exists():
    # Try parent directory (in case running from processor/)
    if (repo_root.parent / 'processed_maps').exists():
        repo_root = repo_root.parent
        os.chdir(repo_root)
    else:
        print("WARNING: processed_maps/ directory not found!")
        print("   Make sure you are in the correct directory.")
        print(f"   Current directory: {repo_root}")

print(f"Repository root: {repo_root}")
print(f"Setup complete")

NumPy loaded
Matplotlib loaded
SciPy loaded
Pillow loaded

Environment: Local Jupyter
Python: 3.14.0
NumPy: 2.3.5
Matplotlib: 3.10.7

Repository root: c:\Projects\Project_Reality-Mortar-Calculator
Setup complete


## Cell 2: Helper Functions

In [13]:
def load_heightmap(map_path: Path) -> Optional[np.ndarray]:
    """Load heightmap from compressed JSON file.
    
    Args:
        map_path: Path to the map folder containing heightmap.json.gz
        
    Returns:
        2D NumPy array of uint16 values (0-65535), or None if loading failed
    """
    heightmap_file = map_path / 'heightmap.json.gz'
    
    if not heightmap_file.exists():
        print(f"  ERROR: heightmap.json.gz not found at {heightmap_file}")
        return None
    
    try:
        with gzip.open(heightmap_file, 'rt', encoding='utf-8') as f:
            data = json.load(f)
        
        resolution = data['resolution']
        heightmap_1d = np.array(data['data'], dtype=np.uint16)
        heightmap_2d = heightmap_1d.reshape((resolution, resolution))
        
        return heightmap_2d
        
    except Exception as e:
        print(f"  ERROR: Failed to load heightmap: {e}")
        return None


def normalize_elevation(heightmap: np.ndarray, height_scale: float) -> np.ndarray:
    """Convert raw heightmap values to actual elevation in meters.
    
    Formula: elevation_meters = (raw_value / 65535.0) * height_scale
    
    Args:
        heightmap: 2D NumPy array of uint16 values (0-65535)
        height_scale: Maximum elevation from metadata.json (e.g., 300 meters)
        
    Returns:
        2D NumPy array of elevation in meters (float64)
    """
    return (heightmap.astype(np.float64) / 65535.0) * height_scale


def match_minimap_resolution(elevation_array: np.ndarray, target_width: int, target_height: int) -> np.ndarray:
    """Resize elevation array to match minimap resolution using bilinear interpolation.
    
    Args:
        elevation_array: 2D NumPy array of elevation values
        target_width: Target width in pixels
        target_height: Target height in pixels
        
    Returns:
        Resized 2D NumPy array
    """
    current_height, current_width = elevation_array.shape
    
    if current_width == target_width and current_height == target_height:
        return elevation_array
    
    # Calculate zoom factors
    zoom_y = target_height / current_height
    zoom_x = target_width / current_width
    
    # Use scipy.ndimage.zoom with bilinear interpolation (order=1)
    resized = zoom(elevation_array, (zoom_y, zoom_x), order=1)
    
    return resized


def create_terrain_colormap():
    """Create a custom colormap for terrain elevation.
    
    Gradient stops:
    - 0.0: Blue (#0000FF) - sea level / lowest
    - 0.33: Green (#00FF00) - lowland
    - 0.66: Yellow (#FFFF00) - highland
    - 1.0: Red (#FF0000) - peaks / highest
    
    Returns:
        matplotlib LinearSegmentedColormap
    """
    colors = [
        (0.0, 0.0, 1.0),   # Blue at 0.0
        (0.0, 1.0, 0.0),   # Green at 0.33
        (1.0, 1.0, 0.0),   # Yellow at 0.66
        (1.0, 0.0, 0.0)    # Red at 1.0
    ]
    positions = [0.0, 0.33, 0.66, 1.0]
    
    cmap = LinearSegmentedColormap.from_list('terrain_elevation', list(zip(positions, colors)))
    return cmap


print("Helper functions loaded")

Helper functions loaded


## Cell 3: Contour Generation Function

In [14]:
def generate_contour_image(elevation_array: np.ndarray, output_path: Path,
                           target_width: int, target_height: int) -> bool:
    """Generate a contour map image from elevation data.
    
    Features:
    - Filled contour (heat map) background using custom colormap
    - Thin contour lines every 5 meters (linewidth=0.5, dark gray)
    - Thick contour lines every 15 meters (linewidth=1.5, black)
    - Elevation labels on thick lines
    
    Args:
        elevation_array: 2D NumPy array of elevation in meters
        output_path: Path to save the output PNG
        target_width: Output image width in pixels
        target_height: Output image height in pixels
        
    Returns:
        True if successful, False otherwise
    """
    try:
        # Get elevation range
        min_elev = elevation_array.min()
        max_elev = elevation_array.max()
        
        # Calculate contour levels
        # Thin lines every 5 meters
        thin_levels = np.arange(min_elev, max_elev + 5, 5)
        # Thick lines every 15 meters
        thick_levels = np.arange(min_elev, max_elev + 15, 15)
        
        # Create figure with exact pixel dimensions
        dpi = 100
        fig_width = target_width / dpi
        fig_height = target_height / dpi
        
        fig, ax = plt.subplots(figsize=(fig_width, fig_height), dpi=dpi)
        
        # Remove all margins and padding
        ax.set_position([0, 0, 1, 1])
        ax.axis('off')
        
        # Create custom colormap
        terrain_cmap = create_terrain_colormap()
        
        # Draw filled contour (heat map background)
        # Use more levels for smoother color transitions
        fill_levels = np.linspace(min_elev, max_elev, 50)
        ax.contourf(elevation_array, levels=fill_levels, cmap=terrain_cmap)
        
        # Draw thin contour lines (every 5m)
        thin_contours = ax.contour(elevation_array, levels=thin_levels,
                                    colors='#404040', linewidths=0.5, alpha=0.6)
        
        # Draw thick contour lines (every 15m)
        thick_contours = ax.contour(elevation_array, levels=thick_levels,
                                     colors='#000000', linewidths=1.5)
        
        # Add elevation labels on thick lines
        # Use inline labels, ASCII-only format, smaller font
        ax.clabel(thick_contours, inline=True, fontsize=8, fmt='%d',
                  inline_spacing=5, use_clabeltext=True)
        
        # Set axis limits to match data exactly
        ax.set_xlim(0, elevation_array.shape[1])
        ax.set_ylim(elevation_array.shape[0], 0)  # Flip Y axis to match image coordinates
        
        # Save as PNG
        fig.savefig(output_path, format='png', dpi=dpi, pad_inches=0,
                    bbox_inches='tight', transparent=False)
        plt.close(fig)
        
        # Flip the image vertically to match minimap orientation
        # (PR minimap has Y-axis increasing downward, matching image coordinates)
        with Image.open(output_path) as img:
            flipped_img = img.transpose(Image.FLIP_TOP_BOTTOM)
            flipped_img.save(output_path, 'PNG')
        
        return True
        
    except Exception as e:
        print(f"  ERROR: Failed to generate contour image: {e}")
        return False


def update_metadata(map_path: Path, contour_info: dict) -> bool:
    """Update metadata.json with contour map information.
    
    Args:
        map_path: Path to the map folder
        contour_info: Dictionary containing contour map metadata
        
    Returns:
        True if successful, False otherwise
    """
    metadata_path = map_path / 'metadata.json'
    
    try:
        # Read existing metadata
        with open(metadata_path, 'r', encoding='utf-8') as f:
            metadata = json.load(f)
        
        # Add contourmap field
        metadata['contourmap'] = contour_info
        
        # Write updated metadata
        with open(metadata_path, 'w', encoding='utf-8') as f:
            json.dump(metadata, f, indent=2, ensure_ascii=False)
        
        return True
        
    except Exception as e:
        print(f"  ERROR: Failed to update metadata: {e}")
        return False


print("Contour generation functions loaded")

Contour generation functions loaded


## Cell 4: Discover Maps

In [15]:
# Test: Regenerate contour map for one map to verify Y-axis fix
test_map_path = repo_root / 'processed_maps' / 'muttrah_city_2'

# Load metadata
with open(test_map_path / 'metadata.json', 'r', encoding='utf-8') as f:
    metadata = json.load(f)

height_scale = metadata.get('height_scale', 300)
print(f"Testing with map: muttrah_city_2")
print(f"Height scale: {height_scale}m")

# Get minimap resolution
minimap_path = test_map_path / 'minimap.png'
with Image.open(minimap_path) as img:
    target_width, target_height = img.size
print(f"Minimap resolution: {target_width}x{target_height}")

# Load heightmap
heightmap = load_heightmap(test_map_path)
print(f"Heightmap: {heightmap.shape[0]}x{heightmap.shape[1]}")

# Normalize to meters
elevation = normalize_elevation(heightmap, height_scale)
print(f"Elevation range: {elevation.min():.1f}m - {elevation.max():.1f}m")

# Resize to match minimap
elevation_resized = match_minimap_resolution(elevation, target_width, target_height)
print(f"Resized to: {elevation_resized.shape[1]}x{elevation_resized.shape[0]}")

# Generate contour image (with Y-axis flip fix)
output_path = test_map_path / 'contourmap.png'
if generate_contour_image(elevation_resized, output_path, target_width, target_height):
    print(f"OK: Contour map regenerated with Y-axis fix!")
    output_size_kb = output_path.stat().st_size / 1024
    print(f"File size: {output_size_kb:.1f} KB")
else:
    print("ERROR: Failed to generate contour image")

Testing with map: muttrah_city_2
Height scale: 300m
Minimap resolution: 2048x2048
Heightmap: 1025x1025
Heightmap: 1025x1025
Elevation range: 0.0m - 298.8m
Resized to: 2048x2048
OK: Contour map regenerated with Y-axis fix!
File size: 3400.6 KB


In [16]:
# Discover all map folders in processed_maps/
processed_dir = repo_root / 'processed_maps'

map_folders = []
for item in sorted(processed_dir.iterdir()):
    if item.is_dir():
        # Check if map has required files
        has_heightmap = (item / 'heightmap.json.gz').exists()
        has_metadata = (item / 'metadata.json').exists()
        
        if has_heightmap and has_metadata:
            map_folders.append(item)
        else:
            missing = []
            if not has_heightmap:
                missing.append('heightmap.json.gz')
            if not has_metadata:
                missing.append('metadata.json')
            print(f"WARNING: Skipping {item.name} - missing: {', '.join(missing)}")

print(f"\nFound {len(map_folders)} maps to process")

# Check minimap availability
maps_with_minimap = sum(1 for m in map_folders if (m / 'minimap.png').exists())
print(f"  {maps_with_minimap} have minimap.png (will use its resolution)")
print(f"  {len(map_folders) - maps_with_minimap} without minimap (will use default resolution)")


Found 83 maps to process
  83 have minimap.png (will use its resolution)
  0 without minimap (will use default resolution)


## Cell 5: Processing Loop

In [17]:
# Processing statistics
stats = {
    'total': len(map_folders),
    'processed': 0,
    'errors': 0,
    'error_maps': []
}

# Default resolution if no minimap exists
DEFAULT_RESOLUTION = 2048

print(f"{'='*70}")
print(f"Processing {stats['total']} maps...")
print(f"{'='*70}\n")

start_time = datetime.now()

for i, map_path in enumerate(map_folders, 1):
    map_name = map_path.name
    print(f"[{i}/{stats['total']}] {map_name}")
    
    try:
        # Load metadata for height_scale
        with open(map_path / 'metadata.json', 'r', encoding='utf-8') as f:
            metadata = json.load(f)
        
        height_scale = metadata.get('height_scale', 300)
        print(f"  Height scale: {height_scale}m")
        
        # Determine target resolution from minimap.png
        minimap_path = map_path / 'minimap.png'
        if minimap_path.exists():
            with Image.open(minimap_path) as img:
                target_width, target_height = img.size
            print(f"  Minimap resolution: {target_width}x{target_height}")
        else:
            target_width = DEFAULT_RESOLUTION
            target_height = DEFAULT_RESOLUTION
            print(f"  No minimap, using default: {target_width}x{target_height}")
        
        # Load heightmap
        print(f"  Loading heightmap...")
        heightmap = load_heightmap(map_path)
        if heightmap is None:
            raise Exception("Failed to load heightmap")
        print(f"  Heightmap: {heightmap.shape[0]}x{heightmap.shape[1]}")
        
        # Normalize to meters
        print(f"  Normalizing elevation...")
        elevation = normalize_elevation(heightmap, height_scale)
        print(f"  Elevation range: {elevation.min():.1f}m - {elevation.max():.1f}m")
        
        # Resize to match minimap
        print(f"  Matching minimap resolution...")
        elevation_resized = match_minimap_resolution(elevation, target_width, target_height)
        print(f"  Resized to: {elevation_resized.shape[1]}x{elevation_resized.shape[0]}")
        
        # Generate contour image
        print(f"  Generating contour map...")
        output_path = map_path / 'contourmap.png'
        if not generate_contour_image(elevation_resized, output_path, target_width, target_height):
            raise Exception("Failed to generate contour image")
        
        # Verify output
        output_size_kb = output_path.stat().st_size / 1024
        print(f"  Contour map saved: {output_size_kb:.1f} KB")
        
        # Update metadata
        print(f"  Updating metadata...")
        contour_info = {
            'file': 'contourmap.png',
            'resolution': f"{target_width}x{target_height}",
            'generated_at': datetime.utcnow().isoformat() + 'Z',
            'thin_interval_m': 5,
            'thick_interval_m': 15
        }
        
        if not update_metadata(map_path, contour_info):
            raise Exception("Failed to update metadata")
        
        stats['processed'] += 1
        print(f"  OK: {map_name} processed successfully\n")
        
    except Exception as e:
        stats['errors'] += 1
        stats['error_maps'].append(map_name)
        print(f"  ERROR: {e}\n")

end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

print(f"{'='*70}")
print(f"Processing complete!")
print(f"  Processed: {stats['processed']}/{stats['total']}")
print(f"  Errors: {stats['errors']}")
print(f"  Duration: {duration:.1f} seconds ({duration/60:.1f} minutes)")
print(f"{'='*70}\n")

if stats['error_maps']:
    print("Maps with errors:")
    for map_name in stats['error_maps']:
        print(f"  - {map_name}")

Processing 83 maps...

[1/83] adak
  Height scale: 300m
  Minimap resolution: 2048x2048
  Loading heightmap...
  Heightmap: 1025x1025
  Normalizing elevation...
  Elevation range: 0.0m - 219.3m
  Matching minimap resolution...
  Heightmap: 1025x1025
  Normalizing elevation...
  Elevation range: 0.0m - 219.3m
  Matching minimap resolution...
  Resized to: 2048x2048
  Generating contour map...
  Resized to: 2048x2048
  Generating contour map...


  Contour map saved: 1711.2 KB
  Updating metadata...
  OK: adak processed successfully

[2/83] albasrah_2
  Height scale: 300m
  Minimap resolution: 2048x2048
  Loading heightmap...


C:\Users\lenovo\AppData\Local\Temp\ipykernel_16552\3270431775.py:73: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'generated_at': datetime.utcnow().isoformat() + 'Z',


  Heightmap: 1025x1025
  Normalizing elevation...
  Elevation range: 0.0m - 61.8m
  Matching minimap resolution...
  Resized to: 2048x2048
  Generating contour map...
  Contour map saved: 1474.5 KB
  Updating metadata...
  OK: albasrah_2 processed successfully

[3/83] andromeda
  Height scale: 300m
  Minimap resolution: 2048x2048
  Loading heightmap...
  Heightmap: 513x513
  Normalizing elevation...
  Elevation range: 45.8m - 45.8m
  Matching minimap resolution...
  Resized to: 2048x2048
  Generating contour map...
  ERROR: Failed to generate contour image: Contour levels must be increasing
  ERROR: Failed to generate contour image

[4/83] asad_khal
  Height scale: 300m
  Minimap resolution: 2048x2048
  Loading heightmap...
  Heightmap: 513x513
  Normalizing elevation...
  Elevation range: 82.5m - 142.1m
  Matching minimap resolution...
  Resized to: 2048x2048
  Generating contour map...
  Contour map saved: 1057.4 KB
  Updating metadata...
  OK: asad_khal processed successfully

[5/83

## Cell 6: Summary

In [18]:
print(f"\n{'='*70}")
print("PROCESSING SUMMARY")
print(f"{'='*70}\n")

print(f"Maps processed: {stats['processed']}/{stats['total']}")
print(f"Errors: {stats['errors']}")

if stats['error_maps']:
    print(f"\nMaps with errors:")
    for map_name in stats['error_maps']:
        print(f"  - {map_name}")

# Calculate total contourmap size
total_size = 0
contourmap_count = 0
for map_dir in processed_dir.iterdir():
    if map_dir.is_dir():
        contourmap = map_dir / 'contourmap.png'
        if contourmap.exists():
            total_size += contourmap.stat().st_size
            contourmap_count += 1

total_size_mb = total_size / (1024 * 1024)
print(f"\nTotal contourmap.png files: {contourmap_count}")
print(f"Total contourmap size: {total_size_mb:.1f} MB")
print(f"Output directory: {processed_dir}")

print(f"\n{'='*70}")
print("NEXT STEPS")
print(f"{'='*70}\n")

if stats['processed'] > 0:
    print("1. OK: Contour maps generated successfully")
    print("2. OK: metadata.json updated for all processed maps")
    print("3. -> Commit changes: git add processed_maps/ && git commit -m 'feat: add contour maps'")
    print("4. -> Test the calculator: run calculator/server.py")
    print("5. -> Toggle 'Contour Map' checkbox to verify overlay")
else:
    print("WARNING: No maps were processed.")
    print("  Check errors above and verify processed_maps/ contains valid map folders.")

print(f"\n{'='*70}")
print("Notebook execution complete!")
print(f"{'='*70}")


PROCESSING SUMMARY

Maps processed: 81/83
Errors: 2

Maps with errors:
  - andromeda
  - shipment

Total contourmap.png files: 81
Total contourmap size: 188.0 MB
Output directory: c:\Projects\Project_Reality-Mortar-Calculator\processed_maps

NEXT STEPS

1. OK: Contour maps generated successfully
2. OK: metadata.json updated for all processed maps
3. -> Commit changes: git add processed_maps/ && git commit -m 'feat: add contour maps'
4. -> Test the calculator: run calculator/server.py
5. -> Toggle 'Contour Map' checkbox to verify overlay

Notebook execution complete!
